# **Tabela Bronze**

Obtenção dos dados e envio para o banco de dados

In [0]:
import pandas as pd
from io import BytesIO

def buscar_arquivos_raw(nome_arquivo, pasta_origem=RAW_FOLDER):
    """
    Busca todos os arquivos com o nome informado dentro da pasta raw.
    Exemplo:
        buscar_arquivos_raw("ecommerce_enderecos.parquet")
    """
    arquivos = [
        p.name
        for p in file_system_client.get_paths(path=pasta_origem)
        if p.name.endswith(nome_arquivo)
    ]

    arquivos = sorted(arquivos)

    print(f"Arquivos encontrados para {nome_arquivo}: {len(arquivos)}")
    for arquivo in arquivos:
        print(arquivo)

    return arquivos


def ler_parquets_adls(lista_arquivos):
    """
    Lê uma lista de arquivos Parquet do ADLS e une tudo em um único DataFrame Pandas.
    Também adiciona a coluna arquivo_origem para rastreabilidade da bronze.
    """
    if not lista_arquivos:
        raise ValueError("A lista de arquivos está vazia. Verifique o nome do arquivo e a pasta RAW_FOLDER.")

    dfs = []

    for arquivo in lista_arquivos:
        file_client = file_system_client.get_file_client(arquivo)
        conteudo = file_client.download_file().readall()

        df_temp = pd.read_parquet(BytesIO(conteudo))
        df_temp["arquivo_origem"] = arquivo

        dfs.append(df_temp)

    df_final = pd.concat(dfs, ignore_index=True)

    print("DataFrame consolidado:", df_final.shape)
    return df_final


## Ler `ecommerce_rastreamento.parquet` e consolidar


In [0]:
arquivos_rastreamento = buscar_arquivos_raw("ecommerce_rastreamento.parquet")

df_rastreamento = ler_parquets_adls(arquivos_rastreamento)

display(df_rastreamento.head())
print("Colunas:", df_rastreamento.columns.tolist())


## Converter para Spark DataFrame


In [0]:
df_rastreamento_spark = spark.createDataFrame(df_rastreamento)

df_rastreamento_spark.printSchema()
print("Total de registros:", df_rastreamento_spark.count())

display(df_rastreamento_spark.limit(5))


Percentual de preenchimento

In [0]:
(df_rastreamento.isnull().sum() / len(df_rastreamento) * 100).sort_values(ascending=False)

## Funções de escrita segura no SQL Server


In [0]:
def tabela_existe(spark, jdbc_url, connection_properties, schema, tabela):
    query = f"""
    (
        SELECT COUNT(*) AS qtd
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{schema}'
          AND TABLE_NAME = '{tabela}'
    ) x
    """

    qtd = (
        spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=connection_properties
        )
        .collect()[0]["qtd"]
    )

    return qtd > 0


def colunas_tabela(spark, jdbc_url, connection_properties, schema, tabela):
    # Não usar ORDER BY aqui. Em subquery JDBC do SQL Server isso gera erro.
    query = f"""
    (
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{schema}'
          AND TABLE_NAME = '{tabela}'
    ) x
    """

    df_cols = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_properties
    )

    return [row["COLUMN_NAME"] for row in df_cols.collect()]


def escrever_sqlserver_seguro(
    df_spark,
    spark,
    jdbc_url,
    connection_properties,
    jdbc_hostname,
    jdbc_database,
    jdbc_username,
    jdbc_password,
    schema,
    tabela
):
    """
    Se a tabela não existir, tenta criá-la usando append.
    Se existir, só faz append quando a quantidade e os nomes das colunas forem compatíveis.
    """
    tabela_destino = f"{schema}.{tabela}"

    existe = tabela_existe(
        spark=spark,
        jdbc_url=jdbc_url,
        connection_properties=connection_properties,
        schema=schema,
        tabela=tabela
    )

    if existe:
        colunas_banco = colunas_tabela(
            spark=spark,
            jdbc_url=jdbc_url,
            connection_properties=connection_properties,
            schema=schema,
            tabela=tabela
        )

        colunas_df = df_spark.columns

        colunas_banco_norm = sorted([c.lower() for c in colunas_banco])
        colunas_df_norm = sorted([c.lower() for c in colunas_df])

        if len(colunas_banco) != len(colunas_df) or colunas_banco_norm != colunas_df_norm:
            raise Exception(
                f"""
                Estrutura incompatível para {tabela_destino}.

                Banco: {len(colunas_banco)} colunas
                DataFrame: {len(colunas_df)} colunas

                Colunas banco:
                {colunas_banco}

                Colunas DataFrame:
                {colunas_df}

                Correção:
                - Use outra tabela de destino; ou
                - Ajuste as colunas do DataFrame; ou
                - Recrie a tabela no banco com o schema correto.
                """
            )

        modo = "append"
        print(f"Tabela {tabela_destino} existe e é compatível. Usando append.")

    else:
        modo = "append"
        print(f"Tabela {tabela_destino} não existe. Criando tabela com append.")

    (
        df_spark.write
        .format("sqlserver")
        .mode(modo)
        .option("host", jdbc_hostname)
        .option("port", "1433")
        .option("database", jdbc_database)
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )

    print(f"Carga finalizada em {tabela_destino}. Modo usado: {modo}")


# **Carga para o banco de dados**

In [0]:

# A tabela bronze terá também a coluna arquivo_origem.
escrever_sqlserver_seguro(
    df_spark=df_rastreamento_spark,
    spark=spark,
    jdbc_url=jdbc_url,
    connection_properties=connection_properties,
    jdbc_hostname=JDBC_HOSTNAME,
    jdbc_database=JDBC_DATABASE,
    jdbc_username=JDBC_USERNAME,
    jdbc_password=JDBC_PASSWORD,
    schema="squad1",
    tabela="bronze_ecommerce_rastreamento"
)


## Validar carga no banco

In [0]:
df_validacao_bronze = spark.read.jdbc(
    url=jdbc_url,
    table="squad1.bronze_ecommerce_rastreamento",
    properties=connection_properties
)

print("Total no banco:", df_validacao_bronze.count())
display(df_validacao_bronze.limit(5))
